# HTCTTL-02 — Phân tích CFA/CB-SEM bằng R (lavaan)

Notebook chạy lại toàn bộ quy trình phân tích và vẽ hình giống bản local.

**Cách dùng:** `Runtime → Run all` (hoặc chạy lần lượt từng ô). Các bước nặng (bootstrap, CFA/SEM) có thể mất vài phút đến vài chục phút. Kết quả nằm trong thư mục `/content/htcttl02-colab/htcttl02-R-phantich` (mất khi phiên Colab bị đóng — hãy tải về ở ô cuối).


## 0. Lấy file từ GitHub

In [ ]:
PKG <- "/content/htcttl02-colab/htcttl02-R-phantich"
if (!dir.exists(PKG)) {
  system("git clone --depth 1 -b main https://github.com/saodonga/htcttl02-colab.git /content/htcttl02-colab")
}
stopifnot(dir.exists(PKG))
T0 <- Sys.time()
cat("Gói nằm tại:", PKG, "\n")


## 1. Cài thư viện hệ thống (apt)

In [ ]:
system("apt-get -qq update && apt-get -qq install -y graphviz")


## 2. Cài gói R
Dùng kho nhị phân Posit Package Manager để cài nhanh (không phải biên dịch).

In [ ]:
rel <- tryCatch(trimws(system("lsb_release -cs", intern = TRUE)), error = function(e) "jammy")
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", rel)))
need <- setdiff(c("dplyr", "ggplot2", "lavaan", "psych", "readr", "semPlot", "semTools", "tidySEM"), rownames(installed.packages()))
if (length(need)) install.packages(need, Ncpus = 2)
missing <- setdiff(c("dplyr", "ggplot2", "lavaan", "psych", "readr", "semPlot", "semTools", "tidySEM"), rownames(installed.packages()))
if (length(missing)) stop("Chưa cài được: ", paste(missing, collapse = ", "))


## 3. Chạy `data/sem_output/analysis_sem.R`

In [ ]:
old <- setwd(file.path(PKG, "data/sem_output"))
source("analysis_sem.R", echo = TRUE, max.deparse.length = Inf, encoding = "UTF-8")
setwd(old)


## 4. Chạy `data/sem_output/plot_tidySEM_final.R`

In [ ]:
old <- setwd(file.path(PKG, "data/sem_output"))
source("plot_tidySEM_final.R", echo = TRUE, max.deparse.length = Inf, encoding = "UTF-8")
setwd(old)


## 5. Xem các hình vừa tạo

In [ ]:
imgs <- list.files(PKG, pattern = "\\.(png|jpg|jpeg)$", recursive = TRUE, full.names = TRUE)
imgs <- imgs[file.mtime(imgs) >= T0]
for (p in imgs) { cat("\n", sub(paste0(PKG, "/"), "", p, fixed = TRUE), "\n"); IRdisplay::display_png(file = p) }


## 6. Tải KẾT QUẢ về máy
File zip chỉ chứa **kết quả** tạo ra trong phiên này (bảng csv/excel và hình ảnh). Không chứa dữ liệu đầu vào hay mã nguồn. Tải `/content/ket_qua.zip` ở bảng Files (biểu tượng thư mục bên trái).

In [ ]:
excl <- c("seci_sem_data_219_2026-10-05.csv")
fs <- list.files(PKG, pattern = "\\.(csv|xlsx|xls|png|jpg|jpeg)$", recursive = TRUE, ignore.case = TRUE)
fs <- fs[!grepl("^_shared/", fs) & !(basename(fs) %in% excl) & file.mtime(file.path(PKG, fs)) >= T0]
if (!length(fs)) stop("Chưa có file kết quả nào để nén.")
unlink("/content/ket_qua.zip")
old <- setwd(PKG); utils::zip("/content/ket_qua.zip", files = fs, flags = "-q"); setwd(old)
cat("Đã nén", length(fs), "file kết quả vào /content/ket_qua.zip\n")
